In [1]:
import torch
import joblib
import numpy as np
import re
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from pymorphy3 import MorphAnalyzer
import nltk
from nltk.corpus import stopwords


try:
    stopwords.words("russian")
except LookupError:
    nltk.download('stopwords')


TOKENIZER_PATH = './saved_model_specific_best'
MODEL_SPECIFIC_PATH = './saved_model_specific_best'
ENCODER_SPECIFIC_PATH = 'label_encoder_specific.joblib'
MODEL_GENERAL_PATH = './saved_model_general_best'
ENCODER_GENERAL_PATH = 'label_encoder_general.joblib'

ZOD_KEYWORDS = [
    'начать', 'выполнять', 'остановить', 'начинать'
]

EMAIL_KEYWORDS = [
    'отправить', 'отправлять'
]

TEST_KEYWORDS = [
    'тест', 'тестовый', 'тестовое', 'test'
]

morph = MorphAnalyzer() # Создаем объект Pymorphy3
russian_stopwords = stopwords.words("russian")

print("Все переменные и константы инициализированы.")

Все переменные и константы инициализированы.


In [2]:
device = torch.device("cpu")

try:
    tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH)
    print("Токенизатор успешно загружен.")
except Exception as e:
    print(f"Ошибка загрузки токенизатора: {e}")

try:
    # Модель "Специалист" для детальных паттернов
    model_specific = AutoModelForSequenceClassification.from_pretrained(MODEL_SPECIFIC_PATH)
    model_specific.to(device)
    model_specific.eval()  # Перевод модели в режим предсказания
    print("Модель 'Специалист' успешно загружена.")

    # Модель "Сортировщик" для общих классов
    model_general = AutoModelForSequenceClassification.from_pretrained(MODEL_GENERAL_PATH)
    model_general.to(device)
    model_general.eval()  # Перевод модели в режим предсказания
    print("Модель 'Сортировщик' успешно загружена.")
except Exception as e:
    print(f"Ошибка загрузки моделей: {e}")

try:
    encoder_specific = joblib.load(ENCODER_SPECIFIC_PATH)
    print("Кодировщик для 'Специалиста' успешно загружен.")
    
    encoder_general = joblib.load(ENCODER_GENERAL_PATH)
    print("Кодировщик для 'Сортировщика' успешно загружен.")
except Exception as e:
    print(f"Ошибка загрузки кодировщиков: {e}")

print("\nВсе компоненты готовы к работе.")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "D:\projects\context\contextual_search\venv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "D:\projects\context\contextual_search\venv\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "D:\projects\context\contextual_search\venv\Lib\site-packages\ipykernel\kernelapp.py", line 758, in start
    self.io_loop.

Токенизатор успешно загружен.
Модель 'Специалист' успешно загружена.
Модель 'Сортировщик' успешно загружена.
Кодировщик для 'Специалиста' успешно загружен.
Кодировщик для 'Сортировщика' успешно загружен.

Все компоненты готовы к работе.


In [4]:
def preprocess_text_pymorphy(text: str, morph_analyzer: MorphAnalyzer, stopwords_set: set) -> str:
    """
    Быстрая обработка текста с использованием Pymorphy3.
    """
    text = str(text).lower()
    text = re.sub(r'[^а-яА-ЯёЁ\s]', ' ', text)
    
    tokens = text.split()
    lemmatized_tokens = [
        morph_analyzer.parse(word)[0].normal_form
        for word in tokens
        if word not in stopwords_set
    ]
    
    return " ".join(lemmatized_tokens)

def classify_message(raw_text: str, confidence_threshold: float = 0.90):
    """
    Выполняет полную иерархическую классификацию одного сообщения.
    """
    processed_text = preprocess_text_pymorphy(raw_text, morph, russian_stopwords)
    
    if not processed_text: # Если после обработки осталась пустая строка
        return "Неопределено (пусто)"
        
    if any(keyword in processed_text.split() for keyword in TEST_KEYWORDS):
        return "Тестовое"
    
    first_word = processed_text.split()[0]
    if first_word in ZOD_KEYWORDS:
        return "ZOD"
    if first_word in EMAIL_KEYWORDS:
        return "EMAIL"

    inputs = tokenizer(processed_text, return_tensors="pt", truncation=True, padding=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        logits_specific = model_specific(**inputs).logits
    
    probabilities_specific = torch.softmax(logits_specific, dim=1)[0]
    max_prob_specific = torch.max(probabilities_specific)
    pred_idx_specific = torch.argmax(probabilities_specific)

    if max_prob_specific > confidence_threshold:
        return encoder_specific.inverse_transform([pred_idx_specific.item()])[0]
    else:
        with torch.no_grad():
            logits_general = model_general(**inputs).logits
        
        pred_idx_general = torch.argmax(logits_general, dim=1)
        return encoder_general.inverse_transform([pred_idx_general.item()])[0]

print("Функции `preprocess_text_pymorphy` и `classify_message` созданы.")

Функции `preprocess_text_pymorphy` и `classify_message` созданы.


In [5]:
test_messages = [
    # 1. Сообщение, которое должен определить "Специалист" (CPU)
    "Авария 10005\nИС: Monitoring (Zabbix)\nОписание: Высокая загрузка CPU на узле pg-master-01. Текущее значение: 95%. Назначено на группу Operation-DBA.",
    
    # 2. Сообщение, которое должен определить "Специалист" (HTTP 500)
    "Авария 10003\nИС: Deposits Back-Office\nОписание: Недоступность авторизации в API. Код ответа: HTTP 500. Назначено на группу Dev-Team-Alpha.",
    
    # 3. Сообщение для Rule-Based классификации (ZOD)
    "Выполнен: 01.05.25 Выгрузка данных в витрину отчетности по расписанию.",
    
    # 4. Неоднозначное сообщение, где сработает "Сортировщик"
    "Сервис Deposits Back-Office недоступен. Пользователи не могут войти в систему. Проблема расследуется.",
    
    # 5. Сообщение для Rule-Based классификации (Тестовое)
    "тестовое оповещение для проверки работы системы мониторинга",
    
    # 6. Еще один пример для "Специалиста" (Oracle)
    "Авария 10013\nИС: Deposits Back-Office\nОписание: Ошибка соединения с БД. TNS:listener does not currently know of service requested in connect descriptor (ORA-12514)"
]

print("--- Демонстрация работы иерархического классификатора ---\n")

for i, message in enumerate(test_messages):
    predicted_label = classify_message(message)
    print(f"Пример #{i+1}")
    print(f"  Сообщение: \"{message[:100]}...\"") # Обрезаем длинные сообщения для красивого вывода
    print(f"  ---> Предсказанный класс: {predicted_label}\n")

--- Демонстрация работы иерархического классификатора ---

Пример #1
  Сообщение: "Авария 10005
ИС: Monitoring (Zabbix)
Описание: Высокая загрузка CPU на узле pg-master-01. Текущее зн..."
  ---> Предсказанный класс: Превышение порогов (Мониторинг)

Пример #2
  Сообщение: "Авария 10003
ИС: Deposits Back-Office
Описание: Недоступность авторизации в API. Код ответа: HTTP 50..."
  ---> Предсказанный класс: API Авторизации (HTTP 500)

Пример #3
  Сообщение: "Выполнен: 01.05.25 Выгрузка данных в витрину отчетности по расписанию...."
  ---> Предсказанный класс: ZOD

Пример #4
  Сообщение: "Сервис Deposits Back-Office недоступен. Пользователи не могут войти в систему. Проблема расследуется..."
  ---> Предсказанный класс: Сетевая недоступность (TLS/HTTPS/Ping)

Пример #5
  Сообщение: "тестовое оповещение для проверки работы системы мониторинга..."
  ---> Предсказанный класс: Тестовое

Пример #6
  Сообщение: "Авария 10013
ИС: Deposits Back-Office
Описание: Ошибка соединения с БД. TNS:listener do